# Length generalization in a small arithmetic transformer

A ~10M-parameter decoder-only transformer is trained on 3-digit addition only, then evaluated on operands with 1–6 digits. Three variants are compared:

| Variant | Positional encoding | Answer order |
|---|---|---|
| `baseline` | learned | natural (`579`) |
| `reversed` | learned | reversed (`975`) |
| `nope`     | none    | reversed |

The hypothesis: **learned positional embeddings can't extrapolate** — positions never seen at training time stay at random init, so any sequence longer than the training distribution fails catastrophically. NoPE (no positional encoding) leans on the causal-mask asymmetry alone and tends to extrapolate further.

**Runtime on a Colab T4:** ~12–18 minutes for the full sweep at the settings below.

## 1. Setup

Clones the repo (if running on Colab), installs dependencies, and verifies GPU availability.

In [ ]:
import os, sys, subprocess, pathlib

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not pathlib.Path("jax-transformer").exists():
    subprocess.run(["git", "clone", "--quiet",
                    "https://github.com/sananthanarayan/jax-transformer.git"],
                   check=True)

REPO_ROOT = pathlib.Path("jax-transformer" if IN_COLAB else "..").resolve()
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

if IN_COLAB:
    subprocess.run(["pip", "install", "-q", "-U", "flax>=0.10.0", "optax>=0.2.3"], check=True)

import jax
print("jax devices:", jax.devices())

## 2. Smoke test

Builds the model, runs one forward pass, and prints the untrained loss. Should match `ln(vocab_size) ≈ 2.71`.

In [ ]:
import jax.numpy as jnp
import numpy as np
from flax import nnx

from addition_transformer.data import build_arrays, generate_pairs, max_len_for
from addition_transformer.model import Transformer, TransformerConfig, count_params
from addition_transformer.vocab import VOCAB_SIZE

cfg = TransformerConfig()
model = Transformer(cfg, rngs=nnx.Rngs(0))
print(f"params: {count_params(model)/1e6:.2f}M  pos_encoding={cfg.pos_encoding}  max_len={cfg.max_len}")

pairs = generate_pairs("addition", max_digits=3)[:8]
inp, tgt, mask = build_arrays(pairs, "addition", max_len=cfg.max_len)
logits = model(jnp.asarray(inp))
log_probs = jax.nn.log_softmax(logits, axis=-1)
tgt_lp = jnp.take_along_axis(log_probs, jnp.asarray(tgt)[..., None], axis=-1).squeeze(-1)
loss = (-tgt_lp * jnp.asarray(mask)).sum() / max(int(jnp.asarray(mask).sum()), 1)
print(f"untrained masked loss = {float(loss):.4f}  (random baseline = {np.log(VOCAB_SIZE):.4f})")

## 3. Train the three variants

Each variant trains on 3-digit addition only. We use **4 epochs** here instead of the default 8 to keep the Colab session short; the qualitative picture is the same. If you have time to spare, bump `EPOCHS` up.

In [ ]:
import time
from addition_transformer.train import train_model
from addition_transformer.eval import eval_at_digits

OP = "addition"
TRAIN_DIGITS = 3
EVAL_DIGITS = list(range(1, 7))
MODEL_MAX_LEN = max_len_for(OP, max(EVAL_DIGITS))
EPOCHS = 4
EVAL_SAMPLES = 300
SEED = 0

VARIANTS = [
    {"name": "baseline", "label": "Learned PE + natural order",   "pos_encoding": "learned", "reverse_answer": False},
    {"name": "reversed", "label": "Learned PE + reversed answers", "pos_encoding": "learned", "reverse_answer": True},
    {"name": "nope",     "label": "NoPE + reversed answers",       "pos_encoding": "none",    "reverse_answer": True},
]

results = {}
for v in VARIANTS:
    print(f"\n=== {v['name']}: {v['label']} ===")
    t0 = time.time()
    model = train_model(
        op=OP,
        max_digits=TRAIN_DIGITS,
        epochs=EPOCHS,
        seed=SEED,
        reverse_answer=v["reverse_answer"],
        pos_encoding=v["pos_encoding"],
        model_max_len=MODEL_MAX_LEN,
        eval_samples=600,
        log_prefix=f"[{v['name']}] ",
    )
    accs = {}
    for d in EVAL_DIGITS:
        acc = eval_at_digits(model, OP, d, n_samples=EVAL_SAMPLES,
                             reverse_answer=v["reverse_answer"], seed=SEED + 1000)
        accs[d] = acc
        print(f"[{v['name']}] digits={d}: {acc*100:.2f}%")
    results[v["name"]] = {"label": v["label"], "accuracies": accs,
                          "train_time_sec": time.time() - t0}

## 4. The headline chart

In [ ]:
import matplotlib.pyplot as plt

COLORS = {"baseline": "#d62728", "reversed": "#ff7f0e", "nope": "#2ca02c"}

fig, ax = plt.subplots(figsize=(7.5, 4.5))
ax.axvspan(EVAL_DIGITS[0] - 0.1, TRAIN_DIGITS + 0.1, alpha=0.08, color="gray",
           label=f"Training distribution (≤{TRAIN_DIGITS} digits)")
for name, r in results.items():
    xs = sorted(r["accuracies"].keys())
    ys = [r["accuracies"][d] * 100 for d in xs]
    ax.plot(xs, ys, marker="o", linewidth=2, markersize=7,
            color=COLORS.get(name, "tab:blue"), label=r["label"])
ax.set_xlabel("Operand digit count")
ax.set_ylabel("Exact-match accuracy (%)")
ax.set_title(f"Length generalization on {OP} (trained on ≤{TRAIN_DIGITS} digits)")
ax.set_xticks(EVAL_DIGITS)
ax.set_ylim(-2, 102)
ax.grid(True, alpha=0.3)
ax.legend(loc="upper right", framealpha=0.95, fontsize=9)
fig.tight_layout()

pathlib.Path("results").mkdir(exist_ok=True)
fig.savefig("results/length_gen.png", dpi=150)
plt.show()

## 5. Sample predictions

Inspect a few examples from the NoPE model at each digit count to see *how* it fails when it does — typically the high-order digits go first.

In [ ]:
from addition_transformer.data import render, sample_pairs_at_digits
from addition_transformer.train import greedy_generate
from addition_transformer.vocab import PAD_ID, VOCAB, encode

# Re-train just NoPE quickly to have a model handle in scope.
nope_model = train_model(op=OP, max_digits=TRAIN_DIGITS, epochs=EPOCHS, seed=SEED,
                         reverse_answer=True, pos_encoding="none",
                         model_max_len=MODEL_MAX_LEN, eval_samples=200,
                         log_prefix="[nope] ", verbose=False)

for d in (3, 4, 5, 6):
    pairs = sample_pairs_at_digits(d, 4, seed=99)
    prompts, plens, expected = [], [], []
    for a, b in pairs:
        pr, ans = render(int(a), int(b), OP, reverse_answer=True)
        prompts.append(encode(pr) + [PAD_ID] * (MODEL_MAX_LEN - len(pr)))
        plens.append(len(pr))
        expected.append(ans)
    out = greedy_generate(nope_model,
                          np.asarray(prompts, dtype=np.int32),
                          np.asarray(plens, dtype=np.int32),
                          MODEL_MAX_LEN)
    print(f"\n{d}-digit examples:")
    for i, (a, b) in enumerate(pairs):
        gen = out[i, plens[i]:]
        pad = np.where(gen == PAD_ID)[0]
        cut = pad[0] if len(pad) else len(gen)
        got = "".join(VOCAB[int(t)] for t in gen[:cut])[::-1]
        exp = expected[i][::-1]
        ok = "✓" if got == exp else "✗"
        print(f"  {ok} {a} + {b} = {got}   (expected {exp})")